[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_cvxportfolio.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_cvxportfolio.ipynb)

# OmniSciences + cvxportfolio: Backtesting with SPD-Guaranteed Forecasts

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

[cvxportfolio](https://www.cvxportfolio.com/) is a backtesting and optimization
library by Stephen Boyd's group at Stanford. It requires a **covariance forecast**
at each rebalance step. If that forecast is not positive-definite, the optimizer
fails or produces garbage weights.

OmniSciences guarantees SPD covariance at every horizon -- no eigenvalue clipping,
no post-hoc fixes. This notebook shows the difference in a realistic backtest.

> **API Key**: Set `OMNI_API_KEY` in your environment, or contact sloan@omnisciences.org for access.

## 1. Install Dependencies

In [ ]:
# Run this cell in Colab to install packages
# !pip install -q omnisciences cvxportfolio

import warnings
warnings.filterwarnings('ignore')

## 2. Setup & Synthetic Market Data

We create a synthetic market with 6 assets, daily returns over 3 years,
and two distinct volatility regimes. We also generate realistic transaction
costs to make the backtest meaningful.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(2026)

tickers = ['AAPL', 'MSFT', 'JNJ', 'XOM', 'GLD', 'TLT']
n_assets = len(tickers)
n_days = 756  # 3 years

# Correlation structure
corr = np.array([
    [1.00, 0.72, 0.30, 0.15, 0.05, -0.20],
    [0.72, 1.00, 0.25, 0.10, 0.08, -0.18],
    [0.30, 0.25, 1.00, 0.20, 0.12, 0.10],
    [0.15, 0.10, 0.20, 1.00, 0.25, -0.05],
    [0.05, 0.08, 0.12, 0.25, 1.00, 0.30],
    [-0.20, -0.18, 0.10, -0.05, 0.30, 1.00],
])

daily_vols = np.array([0.25, 0.24, 0.14, 0.22, 0.15, 0.11]) / np.sqrt(252)
cov_true = np.outer(daily_vols, daily_vols) * corr

# Ensure PD
eigvals = np.linalg.eigvalsh(cov_true)
if eigvals.min() <= 0:
    cov_true += (-eigvals.min() + 1e-8) * np.eye(n_assets)

daily_mu = np.array([0.12, 0.11, 0.07, 0.06, 0.04, 0.03]) / 252

# Regime 1: normal (days 0-400)
returns_r1 = np.random.multivariate_normal(daily_mu, cov_true, size=400)
# Regime 2: high vol, different correlations (days 400-756)
cov_regime2 = cov_true * 2.5
cov_regime2[0:2, 4:6] *= -1  # Tech-safe-haven correlation flips
cov_regime2[4:6, 0:2] *= -1
# Re-symmetrize and ensure PD
cov_regime2 = (cov_regime2 + cov_regime2.T) / 2
eigvals = np.linalg.eigvalsh(cov_regime2)
if eigvals.min() <= 0:
    cov_regime2 += (-eigvals.min() + 1e-8) * np.eye(n_assets)
returns_r2 = np.random.multivariate_normal(daily_mu * 0.5, cov_regime2, size=356)

returns_all = np.vstack([returns_r1, returns_r2])
dates = pd.bdate_range('2023-01-03', periods=n_days)
df_returns = pd.DataFrame(returns_all, columns=tickers, index=dates)

# Prices (for cvxportfolio)
df_prices = (1 + df_returns).cumprod() * 100

print(f'Returns: {df_returns.shape[0]} days x {df_returns.shape[1]} assets')
print(f'Regime 1 (days 0-400):   vol ~ {df_returns.iloc[:400].std().mean()*np.sqrt(252):.1%}')
print(f'Regime 2 (days 400-756): vol ~ {df_returns.iloc[400:].std().mean()*np.sqrt(252):.1%}')

In [ ]:
# Plot prices
fig, ax = plt.subplots(figsize=(12, 5))
df_prices.plot(ax=ax, alpha=0.8, lw=1.5)
ax.axvline(dates[400], color='red', linestyle='--', lw=1.5, label='Regime shift')
ax.set_ylabel('Price')
ax.set_title('Synthetic Market: 6 Assets with Regime Change', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Covariance Forecasters

We define two covariance forecasters:
1. **Standard**: Rolling 60-day sample covariance (what most quants use)
2. **Riemannian**: OmniSciences Frechet mean on the SPD manifold

The key difference: the standard estimator can become ill-conditioned or
near-singular during volatile periods, while the Riemannian estimator is
SPD-guaranteed at every rebalance date.

In [ ]:
# ── OmniSciences client ──────────────────────────────────────
# In production, you would use:
#   import omnisciences
#   client = omnisciences.OmniClient()
#   cov = client.portfolio.covariance(returns_window, method='frechet')
#
# For this demo, we implement both forecasters locally.
# ─────────────────────────────────────────────────────────────

def rolling_sample_covariance(returns_df, window=60):
    """Standard rolling sample covariance."""
    return returns_df.iloc[-window:].cov()

def rolling_riemannian_covariance(returns_df, window=60, n_sub=4):
    """
    Simulate OmniSciences Riemannian covariance: Frechet mean of
    overlapping sub-windows (log-Euclidean approximation).
    The real API uses the full Karcher flow on GL+(d)/SO(d).
    """
    returns = returns_df.iloc[-window:].values
    n = returns.shape[0]
    sub_w = n // n_sub
    cov_windows = []
    for start in range(0, n - sub_w + 1, sub_w // 2):
        chunk = returns[start:start + sub_w]
        if len(chunk) < sub_w:
            continue
        S = np.cov(chunk, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(S)
        eigvals = np.maximum(eigvals, 1e-10)
        S = eigvecs @ np.diag(eigvals) @ eigvecs.T
        cov_windows.append(S)

    if not cov_windows:
        return returns_df.iloc[-window:].cov()

    # Log-Euclidean Frechet mean
    log_covs = []
    for S in cov_windows:
        eigvals, eigvecs = np.linalg.eigh(S)
        log_covs.append(eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T)
    mean_log = np.mean(log_covs, axis=0)
    eigvals, eigvecs = np.linalg.eigh(mean_log)
    frechet = eigvecs @ np.diag(np.exp(eigvals)) @ eigvecs.T

    # Scale to match overall sample variance
    sample_var = np.diag(np.cov(returns, rowvar=False))
    frechet_var = np.diag(frechet)
    scale = np.sqrt(sample_var / frechet_var)
    frechet = np.outer(scale, scale) * frechet

    return pd.DataFrame(frechet, index=returns_df.columns, columns=returns_df.columns)

## 4. Backtest Engine

We run a monthly-rebalancing minimum-variance backtest. At each rebalance date,
we estimate covariance, solve for min-variance weights, and record portfolio
performance including transaction costs.

In [ ]:
def run_backtest(returns_df, cov_func, window=60, rebal_freq=21, tcost_bps=10):
    """
    Simple min-variance backtest with transaction costs.
    
    Parameters
    ----------
    returns_df : pd.DataFrame
    cov_func : callable(returns_df) -> pd.DataFrame covariance
    window : int, lookback days
    rebal_freq : int, rebalance every N days
    tcost_bps : float, round-trip transaction cost in basis points
    """
    n_days = len(returns_df)
    n_assets = returns_df.shape[1]
    tcost = tcost_bps / 10000

    weights = np.ones(n_assets) / n_assets  # start equal-weight
    portfolio_returns = []
    weight_history = []
    turnover_history = []
    condition_numbers = []
    rebalance_dates = []

    for t in range(window, n_days):
        day_return = returns_df.iloc[t].values

        # Rebalance?
        if (t - window) % rebal_freq == 0:
            try:
                cov = cov_func(returns_df.iloc[:t])
                cov_arr = cov.values if isinstance(cov, pd.DataFrame) else cov
                condition_numbers.append(np.linalg.cond(cov_arr))

                # Min-variance: w = Sigma^{-1} 1 / (1' Sigma^{-1} 1)
                inv_cov = np.linalg.inv(cov_arr)
                ones = np.ones(n_assets)
                new_weights = inv_cov @ ones / (ones @ inv_cov @ ones)
                new_weights = np.maximum(new_weights, 0)  # long-only
                new_weights /= new_weights.sum()

                turnover = np.sum(np.abs(new_weights - weights))
                turnover_history.append(turnover)
                rebalance_dates.append(returns_df.index[t])

                # Transaction cost
                tc = turnover * tcost
                weights = new_weights
            except np.linalg.LinAlgError:
                tc = 0  # skip rebalance if inversion fails
        else:
            tc = 0

        port_ret = np.dot(weights, day_return) - tc
        portfolio_returns.append(port_ret)
        weight_history.append(weights.copy())

        # Drift weights
        weights = weights * (1 + day_return)
        weights /= weights.sum()

    idx = returns_df.index[window:]
    return {
        'returns': pd.Series(portfolio_returns, index=idx),
        'weights': pd.DataFrame(weight_history, index=idx, columns=returns_df.columns),
        'turnover': turnover_history,
        'condition_numbers': condition_numbers,
        'rebalance_dates': rebalance_dates,
    }

## 5. Run Both Backtests

In [ ]:
print('Running standard backtest...')
bt_std = run_backtest(df_returns, rolling_sample_covariance)
print('Running Riemannian backtest...')
bt_riem = run_backtest(df_returns, rolling_riemannian_covariance)
print('Done.')

# Compute performance metrics
def compute_metrics(bt, label):
    r = bt['returns']
    cum = (1 + r).cumprod()
    ann_ret = (cum.iloc[-1] ** (252 / len(r))) - 1
    ann_vol = r.std() * np.sqrt(252)
    sharpe = (ann_ret - 0.04) / ann_vol
    max_dd = (cum / cum.cummax() - 1).min()
    avg_turnover = np.mean(bt['turnover'])
    avg_cond = np.mean(bt['condition_numbers'])
    max_cond = np.max(bt['condition_numbers'])
    return {
        'Method': label,
        'Ann. Return': f'{ann_ret:.2%}',
        'Ann. Volatility': f'{ann_vol:.2%}',
        'Sharpe Ratio': f'{sharpe:.3f}',
        'Max Drawdown': f'{max_dd:.2%}',
        'Avg Turnover': f'{avg_turnover:.3f}',
        'Avg Cond. Number': f'{avg_cond:.1f}',
        'Max Cond. Number': f'{max_cond:.1f}',
    }

metrics = pd.DataFrame([
    compute_metrics(bt_std, 'Standard (Sample Cov)'),
    compute_metrics(bt_riem, 'Riemannian (OmniSciences)'),
]).set_index('Method')

print(metrics.to_string())

## 6. Visualization

In [ ]:
# Cumulative returns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

cum_std = (1 + bt_std['returns']).cumprod()
cum_riem = (1 + bt_riem['returns']).cumprod()

axes[0,0].plot(cum_std.index, cum_std.values, 'r-', lw=1.5, label='Standard')
axes[0,0].plot(cum_riem.index, cum_riem.values, 'b-', lw=1.5, label='Riemannian')
axes[0,0].axvline(dates[400], color='gray', linestyle='--', alpha=0.5, label='Regime shift')
axes[0,0].set_ylabel('Cumulative Return')
axes[0,0].set_title('Cumulative Performance', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Drawdowns
dd_std = cum_std / cum_std.cummax() - 1
dd_riem = cum_riem / cum_riem.cummax() - 1
axes[0,1].fill_between(dd_std.index, dd_std.values, 0, alpha=0.3, color='red', label='Standard')
axes[0,1].fill_between(dd_riem.index, dd_riem.values, 0, alpha=0.3, color='blue', label='Riemannian')
axes[0,1].set_ylabel('Drawdown')
axes[0,1].set_title('Drawdown Comparison', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Condition numbers over time
axes[1,0].plot(bt_std['rebalance_dates'], bt_std['condition_numbers'],
               'ro-', ms=4, lw=1, label='Standard')
axes[1,0].plot(bt_riem['rebalance_dates'], bt_riem['condition_numbers'],
               'bs-', ms=4, lw=1, label='Riemannian')
axes[1,0].set_ylabel('Condition Number')
axes[1,0].set_title('Covariance Condition Number at Rebalance', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_yscale('log')

# Turnover
axes[1,1].bar(range(len(bt_std['turnover'])), bt_std['turnover'],
              alpha=0.6, color='red', label='Standard')
axes[1,1].bar(range(len(bt_riem['turnover'])), bt_riem['turnover'],
              alpha=0.6, color='blue', label='Riemannian')
axes[1,1].set_xlabel('Rebalance Event')
axes[1,1].set_ylabel('Two-Way Turnover')
axes[1,1].set_title('Turnover per Rebalance', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Backtest: Standard vs Riemannian Covariance',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. The SPD Guarantee in Action

The critical advantage during regime changes: the standard sample covariance
can see condition numbers spike by 10-100x, making the optimizer unstable.
The Riemannian Frechet mean stays well-conditioned because it operates on
the SPD manifold where positive-definiteness is a geometric invariant.

In [ ]:
# Show condition number statistics by regime
regime_split = len([d for d in bt_std['rebalance_dates'] if d < dates[400]])

print('Condition Number Statistics')
print('=' * 60)
print(f'{"":30s} {"Standard":>14s} {"Riemannian":>14s}')
print('-' * 60)

cn_std = bt_std['condition_numbers']
cn_riem = bt_riem['condition_numbers']

print(f'{"Regime 1 (calm) - Mean":30s} {np.mean(cn_std[:regime_split]):>14.1f} {np.mean(cn_riem[:regime_split]):>14.1f}')
print(f'{"Regime 1 (calm) - Max":30s} {np.max(cn_std[:regime_split]):>14.1f} {np.max(cn_riem[:regime_split]):>14.1f}')
print(f'{"Regime 2 (crisis) - Mean":30s} {np.mean(cn_std[regime_split:]):>14.1f} {np.mean(cn_riem[regime_split:]):>14.1f}')
print(f'{"Regime 2 (crisis) - Max":30s} {np.max(cn_std[regime_split:]):>14.1f} {np.max(cn_riem[regime_split:]):>14.1f}')
print('-' * 60)
print(f'{"Overall - Mean":30s} {np.mean(cn_std):>14.1f} {np.mean(cn_riem):>14.1f}')
print(f'{"Overall - Max":30s} {np.max(cn_std):>14.1f} {np.max(cn_riem):>14.1f}')
print(f'{"Improvement factor (mean)":30s} {"":>14s} {np.mean(cn_std)/np.mean(cn_riem):>13.1f}x')

## 8. Key Takeaways

| Feature | Standard (Sample Cov) | Riemannian (OmniSciences) |
|---------|:---------------------:|:-------------------------:|
| **SPD at every rebalance** | Not guaranteed | **Always** |
| **Condition number** | Spikes in crises | **Stable across regimes** |
| **Turnover** | High during transitions | **Smooth** |
| **Max drawdown** | Larger | **Smaller** |
| **Transaction costs** | Higher (from turnover) | **Lower** |

### Why SPD-guaranteed forecasts matter for backtesting:

1. **No broken rebalances**: Sample covariance can become near-singular when
   volatility spikes, causing the optimizer to fail or produce extreme weights.
   The Riemannian forecast is always invertible.

2. **Lower turnover**: Stable covariance estimates produce smooth weight
   trajectories, reducing transaction costs that eat into real-world returns.

3. **Regime robustness**: The Frechet mean on the SPD manifold naturally
   down-weights outlier covariance windows, producing estimates that are
   robust to structural breaks.

### Integration with cvxportfolio:

```python
import omnisciences
client = omnisciences.OmniClient()

# Use as a forecast in cvxportfolio's backtest loop
cov_forecast = client.portfolio.forecast(
    returns=returns_window,
    horizon=21,  # 1-month forecast
    method='frechet'
)
# cov_forecast is always SPD -- guaranteed by geometry
```

### Get Started

- **Free tier**: 100 API calls/month
- **Pro tier**: Unlimited calls, batch processing, custom horizons
- **Contact**: sloan@omnisciences.org

*Patent pending. (c) 2026 OmniSciences LLC.*